In [ ]:
import pandas as pd
import numpy as np 
from sqlalchemy import create_engine
import os 
import glob

In [ ]:
def batch_cleaner(data_configuration, Output ="Clean Data"):
    """
    Membersihkan banyak file sekaligus berdasarkan blueprint kolom masing-masing.
    """
    # Buat folder output jika belum ada
    os.makedirs(Output , exist_ok=True)
    
    # Looping membaca konfigurasi tiap file
    for rules_name, spek in data_configuration.items():
        # Cari semua file yang sesuai dengan path (bisa pakai wildcard seperti *.csv)
        file_list = glob.glob(spek['path'])
        
        if not file_list:
            print(f"no files found in path: {spek['path']}")
            continue
            
        for file_path in file_list:
            file_name = os.path.basename(file_path)
            print(f"... Processing: {file_name} ({rules_name})...")
            
            try:
                # 1. Load Data
                df = pd.read_csv(file_path)
                
                # 2. Pembersihan Wajib untuk semua file
                df.columns = df.columns.str.strip()
                df = df.drop_duplicates()
                
                # 3. Pembersihan Spesifik berbasis parameter
                if 'str_column' in spek and spek['str_column']:
                    # Pastikan kolom ada di file sebelum dibersihkan
                    str_target = [c for c in spek['str_column'] if c in df.columns]
                    if str_target:
                        df[str_target] = df[str_target].apply(lambda x: x.astype(str).str.lower().str.strip())
                
                if 'datetime_column' in spek and spek['datetime_column']:
                    for col in spek['datetime_column']:
                        if col in df.columns:
                            df[col] = pd.to_datetime(df[col], errors='coerce')
                
                if 'id_column' in spek and spek['id_column']:
                    target_id = [c for c in spek['id_column'] if c in df.columns]
                    if target_id:
                        df = df.dropna(subset=target_id)
                
                # 4. Ekspor Hasil Bersih
                path_output = os.path.join(Output , f"cleaned_{file_name}")
                df.to_sql(path_output, index=False)
                print(f"saved on: {path_output}")
                
            except Exception as e:
                print(f"failed to process {file_name}. Error: {e}")
                
    print("\n Cleaning Done...")

In [ ]:
data_configuration = {
    'Aturan_Pelanggan': {
        'path': 'folder_mentah/customers.csv', # Bisa diisi satu file spesifik
        'kolom_teks': ['customer_city', 'customer_state'],
        'kolom_id_kritis': ['customer_id']
    },
    
}